####Saving the external locations in variables

In [0]:
%run "/Workspace/Users/soumya.saha221@gmail.com/traffic_data_project/4. Common config"

In [0]:
dbutils.widgets.text(name='env',defaultValue='',label='Enter the Environment')
env = dbutils.widgets.get('env')

####Reading raw traffic bronze table

In [0]:
def read_bronze_table(environment):
    print('Reading raw traffic bronze table: ',end='')
    df = spark.readStream.table(f"{environment}catalog.bronze.raw_traffic")
    print('Success!')
    return df

####Handling duplicate data

In [0]:
def drop_dup(df):
    print('Removing duplicate values')
    df_dedup = df.dropDuplicates()
    print('Duplicates removed')
    return df_dedup


####Handling null values

In [0]:
def handle_nulls(df):
    print('Handling null values')
    columns = df.columns
    df_null = df.fillna('Unknown', subset = columns) #handling string values
    print('Handling string null values')
    df_clean = df_null.fillna(0, subset = columns) #handling integer values
    print('Handling integer null values')
    print('Null values handled')
    return df_clean

####Getting count of total electric vehicles

In [0]:
def ev_count(df):
    print('Calculating EV counts: ',end='')
    from pyspark.sql.functions import col
    df_ev = df.withColumn('EV_total_count', col('EV_car')+col('EV_bike'))
    print('Success!')
    return df_ev

####Getting count of Motor Vehicles

In [0]:
def mv_count(df):
    print('Calculating MV counts: ', end='')
    from pyspark.sql.functions import col
    df_mv = df.withColumn('MV_total_count', col('Two_wheeled_motor_vehicles')+col('Cars_and_taxis') + col('Buses_and_coaches') + col('LGV_type') + col('HGV_type') + col('EV_total_count'))
    print('Success!')
    return df_mv


####Creating transformed time column

In [0]:
def transformed_time(df):
    print('Transforming time column added: ', end='')
    from pyspark.sql.functions import current_timestamp
    df_time  = df.withColumn('Transformed_time', current_timestamp())
    print('Success!')
    return df_time

###Writing to silver traffic table

In [0]:
def write_to_silver(df, environment):
    print('Writing to silver table: ', end='')
    df.writeStream.format('delta')\
    .outputMode('append')\
    .option('checkpointLocation', f"{checkpoint}/silverTrafficLoad/checkpnt")\
      .trigger(availableNow = True) \
    .toTable(f"{environment}catalog.silver.silver_traffic").awaitTermination()
    print('Success!!!')

####Calling all functions

In [0]:
#Read bronze table
df_bronzetraffic = read_bronze_table(env)

#Removing duplicates
df_dup = drop_dup(df_bronzetraffic)

#Handling null values
df_null = handle_nulls(df_dup)

#EV count
df_ev = ev_count(df_null)

#MV count
df_mv = mv_count(df_ev)

#adding transformed time column
df_final = transformed_time(df_mv)

#writing to silver table
write_to_silver(df_final, env)